# Детекция ботов по поведенческим данным

Решение задачи бинарной классификации пользователей на основе событий внутри временного окна.

Целевая переменная:

- `target = 1` — автоматизированный сбор данных;
- `target = 0` — обычный пользователь.

Основная метрика — **Precision при Recall ≥ 0.70**.

Решение основано на поведенческих, временных и исторических признаках. Для финального прогноза используется ансамбль CatBoost и LightGBM.

## 1. Настройка окружения

Импорт библиотек, фиксация `random seed` и настройка путей к данным.

Для воспроизводимости во всех моделях используется фиксированный `SEED = 42`.

In [1]:
from pathlib import Path
import random
import platform
import warnings

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
import lightgbm as lgb

from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA = ROOT / "data"

print("Python:", platform.python_version())
print("Seed:", SEED)

Python: 3.11.9
Seed: 42


In [2]:
from metric import precision_at_recall

TARGET_RECALL = 0.70


def evaluate(y_true, score):
    return {
        "precision_at_recall_70": precision_at_recall(
            y_true,
            score,
            recall=TARGET_RECALL
        ),
        "roc_auc": roc_auc_score(y_true, score),
        "average_precision": average_precision_score(y_true, score),
    }

## 2. Загрузка данных

Используются четыре набора данных:

- `train.csv` — обучающая выборка с целевой переменной;
- `test.csv` — тестовые cookie;
- `events.csv.gz` — журнал пользовательских событий;
- `sample_submission.csv` — шаблон файла с предсказаниями.

Временные поля преобразуются в формат `datetime`, после чего выполняются базовые проверки целостности данных.

In [3]:
train = pd.read_csv(
    DATA / "train.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts"
    ]
)

test = pd.read_csv(
    DATA / "test.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts"
    ]
)

events = pd.read_csv(
    DATA / "events.csv.gz",
    parse_dates=["event_ts"]
)

sample_submission = pd.read_csv(
    "sample_submission.csv"
)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)

display(train.head())

train: (11091, 5)
test: (4909, 4)
events: (328905, 14)


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


## 3. Подготовка событий

Для каждого cookie учитываются только события, произошедшие внутри соответствующего временного окна:

`window_start_ts <= event_ts < window_end_ts`

Это исключает использование событий за пределами периода наблюдения и предотвращает утечку информации из будущего.

In [4]:
assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique

assert set(train["cookie_id"]).isdisjoint(
    set(test["cookie_id"])
)

assert set(train["target"].unique()) == {0, 1}

print("Bots:", int(train["target"].sum()))
print("Humans:", int((train["target"] == 0).sum()))
print("Bot share:", train["target"].mean())

print("Train period:")
print(
    train["window_start_ts"].min(),
    "—",
    train["window_start_ts"].max()
)

print("Test period:")
print(
    test["window_start_ts"].min(),
    "—",
    test["window_start_ts"].max()
)

Bots: 899
Humans: 10192
Bot share: 0.0810567126498963
Train period:
2026-04-06 00:00:00 — 2026-04-19 00:00:00
Test period:
2026-04-20 00:00:00 — 2026-04-26 00:00:00


In [5]:
def select_window_events(meta, events):
    e = events.merge(
        meta[
            [
                "cookie_id",
                "window_start_ts",
                "window_end_ts"
            ]
        ],
        on="cookie_id",
        how="inner",
        validate="many_to_one"
    )

    mask = (
        e["event_ts"].ge(e["window_start_ts"])
        &
        e["event_ts"].lt(e["window_end_ts"])
    )

    return e.loc[mask].copy()

## 4. Предобработка событий

Перед построением признаков выполняется базовая нормализация данных:

- сортировка событий по cookie и времени;
- нормализация платформы;
- выделение признаков из `user_agent`;
- преобразование поисковых параметров;
- вычисление временных интервалов между событиями;
- определение наличия координат указателя.

In [6]:
def prepare_events(meta, events):
    e = select_window_events(meta, events)

    e = e.sort_values(
        ["cookie_id", "event_ts"],
        kind="stable"
    ).copy()

    # Platform
    e["platform_norm"] = (
        e["platform"]
        .fillna("unknown")
        .str.lower()
        .replace({
            "desktop": "web",
            "iphone": "ios"
        })
    )

    # User agent
    ua = (
        e["user_agent"]
        .fillna("")
        .str.lower()
    )

    e["ua_bot"] = ua.str.contains(
        "bot|crawler|spider|python|curl|headless|selenium",
        regex=True
    ).astype("int8")

    e["ua_mobile"] = ua.str.contains(
        "mobile|android|iphone",
        regex=True
    ).astype("int8")

    e["ua_windows"] = ua.str.contains(
        "windows",
        regex=True
    ).astype("int8")

    e["ua_mac"] = ua.str.contains(
        "macintosh|mac os",
        regex=True
    ).astype("int8")

    # Time
    e["hour"] = e["event_ts"].dt.hour
    e["minute"] = e["event_ts"].dt.floor("min")
    e["second"] = e["event_ts"].dt.second

    e["gap"] = (
        e.groupby("cookie_id")["event_ts"]
        .diff()
        .dt.total_seconds()
    )

    # Pointer
    e["has_pointer"] = (
        e["pointer_x"].notna()
        & e["pointer_y"].notna()
    ).astype("int8")

    # Search
    e["query_length"] = (
        e["search_query"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    e["search_page_num"] = pd.to_numeric(
        e["search_page"],
        errors="coerce"
    )

    return e

## 5. Построение признаков

Для каждого cookie формируется набор агрегированных признаков, описывающих поведение пользователя в течение окна наблюдения.

Используются несколько групп признаков:

### Поведенческие признаки

Количество событий, уникальных типов событий, товаров, категорий, локаций, поисковых запросов, платформ и User-Agent.

### Временные признаки

Продолжительность активности, интервалы между событиями, квантили временных интервалов, доля быстрых последовательных действий и повторяющиеся временные интервалы.

### Pointer-признаки

Статистики координат указателя и доля событий, для которых доступны координаты.

### Поисковые признаки

Количество и повторяемость поисковых запросов, длина запросов и статистики страниц поисковой выдачи.

### Event и platform-признаки

Количество и доля различных типов событий и платформ.

### Признаки повторяемости и последовательности

Повторное взаимодействие с одними и теми же товарами, запросами и типами событий.

После объединения этих групп получается базовое представление каждого cookie.

In [7]:
def build_behavior_features(meta, events):
    e = prepare_events(meta, events)

    g = e.groupby("cookie_id", sort=False)

    f = g.agg(
        n_events=("eid", "size"),

        n_event_types=("event_name", "nunique"),
        n_items=("item_id", "nunique"),
        n_categories=("item_category", "nunique"),
        n_locations=("item_location", "nunique"),
        n_queries=("search_query", "nunique"),
        n_platforms=("platform_norm", "nunique"),
        n_agents=("user_agent", "nunique"),

        active_seconds=(
            "event_ts",
            lambda s: (
                s.max() - s.min()
            ).total_seconds()
        ),

        n_minutes=("minute", "nunique"),
        n_hours=("hour", "nunique"),

        gap_mean=("gap", "mean"),
        gap_std=("gap", "std"),
        gap_median=("gap", "median"),
        gap_min=("gap", "min"),
        gap_max=("gap", "max"),

        pointer_rate=("has_pointer", "mean"),

        pointer_x_mean=("pointer_x", "mean"),
        pointer_x_std=("pointer_x", "std"),
        pointer_x_min=("pointer_x", "min"),
        pointer_x_max=("pointer_x", "max"),

        pointer_y_mean=("pointer_y", "mean"),
        pointer_y_std=("pointer_y", "std"),
        pointer_y_min=("pointer_y", "min"),
        pointer_y_max=("pointer_y", "max"),

        query_length_mean=("query_length", "mean"),
        query_length_max=("query_length", "max"),

        search_page_mean=("search_page_num", "mean"),
        search_page_max=("search_page_num", "max"),

        ua_bot_rate=("ua_bot", "mean"),
        ua_mobile_rate=("ua_mobile", "mean"),
        ua_windows_rate=("ua_windows", "mean"),
        ua_mac_rate=("ua_mac", "mean"),

        hour_mean=("hour", "mean"),
        hour_std=("hour", "std"),
    )

    return f

In [8]:
def build_timing_features(meta, events):
    e = prepare_events(meta, events)

    def timing_one_cookie(group):
        gaps = group["gap"].dropna()

        if len(gaps) == 0:
            return pd.Series({
                "gap_q10": 0,
                "gap_q25": 0,
                "gap_q75": 0,
                "gap_q90": 0,
                "gap_iqr": 0,
                "same_second_rate": 0,
                "gap_le_1_rate": 0,
                "gap_le_3_rate": 0,
                "gap_le_10_rate": 0,
                "gap_le_30_rate": 0,
                "gap_cv": 0,
                "repeated_gap_rate": 0
            })

        mean_gap = gaps.mean()

        return pd.Series({
            "gap_q10": gaps.quantile(.10),
            "gap_q25": gaps.quantile(.25),
            "gap_q75": gaps.quantile(.75),
            "gap_q90": gaps.quantile(.90),

            "gap_iqr":
                gaps.quantile(.75)
                - gaps.quantile(.25),

            "same_second_rate":
                (gaps == 0).mean(),

            "gap_le_1_rate":
                (gaps <= 1).mean(),

            "gap_le_3_rate":
                (gaps <= 3).mean(),

            "gap_le_10_rate":
                (gaps <= 10).mean(),

            "gap_le_30_rate":
                (gaps <= 30).mean(),

            "gap_cv":
                gaps.std() / (mean_gap + 1e-6),

            "repeated_gap_rate":
                gaps.duplicated(
                    keep=False
                ).mean()
        })

    timing = (
        e.groupby("cookie_id", sort=False)
        .apply(timing_one_cookie)
    )

    return timing

In [9]:
def build_event_features(meta, events):
    e = prepare_events(meta, events)

    counts = pd.crosstab(
        e["cookie_id"],
        e["event_name"]
    )

    counts.columns = [
        f"event_count__{c}"
        for c in counts.columns
    ]

    rates = counts.div(
        counts.sum(axis=1).replace(0, 1),
        axis=0
    )

    rates.columns = [
        c.replace(
            "event_count__",
            "event_rate__"
        )
        for c in rates.columns
    ]

    return counts.join(
        rates,
        how="outer"
    )

In [10]:
def build_platform_features(meta, events):
    e = prepare_events(meta, events)

    counts = pd.crosstab(
        e["cookie_id"],
        e["platform_norm"]
    )

    counts.columns = [
        f"platform_count__{c}"
        for c in counts.columns
    ]

    rates = counts.div(
        counts.sum(axis=1).replace(0, 1),
        axis=0
    )

    rates.columns = [
        c.replace(
            "platform_count__",
            "platform_rate__"
        )
        for c in rates.columns
    ]

    return counts.join(rates)

In [11]:
def build_repetition_features(meta, events):
    e = prepare_events(meta, events)

    e["repeat_item"] = (
        e["item_id"].notna()
        &
        e.duplicated(
            ["cookie_id", "item_id"]
        )
    )

    e["repeat_query"] = (
        e["search_query"].notna()
        &
        e.duplicated(
            ["cookie_id", "search_query"]
        )
    )

    e["repeat_timestamp"] = e.duplicated(
        ["cookie_id", "event_ts"]
    )

    g = e.groupby("cookie_id")

    result = g.agg(
        repeat_item_rate=("repeat_item", "mean"),
        repeat_query_rate=("repeat_query", "mean"),
        repeat_timestamp_rate=("repeat_timestamp", "mean")
    )

    return result

In [12]:
def build_sequence_features(meta, events):
    e = prepare_events(meta, events)

    e["prev_event"] = (
        e.groupby("cookie_id")["event_name"]
        .shift(1)
    )

    e["same_event_as_previous"] = (
        e["event_name"] == e["prev_event"]
    ).astype("int8")

    e["prev_item"] = (
        e.groupby("cookie_id")["item_id"]
        .shift(1)
    )

    e["same_item_as_previous"] = (
        e["item_id"].notna()
        &
        e["item_id"].eq(e["prev_item"])
    ).astype("int8")

    result = (
        e.groupby("cookie_id")
        .agg(
            consecutive_same_event_rate=(
                "same_event_as_previous",
                "mean"
            ),
            consecutive_same_item_rate=(
                "same_item_as_previous",
                "mean"
            )
        )
    )

    return result

In [13]:
def add_ratio_features(f):
    f = f.copy()

    f["events_per_minute"] = (
        f["n_events"]
        /
        (f["active_seconds"] / 60 + 1)
    )

    f["events_per_active_minute"] = (
        f["n_events"]
        /
        f["n_minutes"].clip(lower=1)
    )

    f["events_per_item"] = (
        f["n_events"]
        /
        f["n_items"].clip(lower=1)
    )

    f["events_per_query"] = (
        f["n_events"]
        /
        f["n_queries"].clip(lower=1)
    )

    f["items_per_minute"] = (
        f["n_items"]
        /
        f["n_minutes"].clip(lower=1)
    )

    f["event_type_diversity"] = (
        f["n_event_types"]
        /
        f["n_events"].clip(lower=1)
    )

    f["item_diversity"] = (
        f["n_items"]
        /
        f["n_events"].clip(lower=1)
    )

    return f

## 6. Исторические признаки товаров

Для усиления модели используются данные о предыдущих наблюдениях.

Для каждого `item_id` рассчитывается статистика его появления у ранее наблюдавшихся пользователей. При построении признаков обучающей выборки используются только cookie, временное окно которых уже завершилось к началу текущего окна.

Таким образом, признаки строятся с соблюдением временного порядка и без использования будущих значений `target`.

In [14]:
def get_cookie_items(meta, events):
    e = select_window_events(meta, events)

    result = (
        e.loc[
            e["item_id"].notna(),
            ["cookie_id", "item_id"]
        ]
        .drop_duplicates()
    )

    return result

In [15]:
def build_train_item_history(
    train_meta,
    events
):
    cookie_items = get_cookie_items(
        train_meta,
        events
    )

    item_map = (
        cookie_items
        .groupby("cookie_id")["item_id"]
        .apply(list)
        .to_dict()
    )

    ordered = (
        train_meta[
            [
                "cookie_id",
                "window_start_ts",
                "target"
            ]
        ]
        .sort_values(
            [
                "window_start_ts",
                "cookie_id"
            ]
        )
    )

    # Статистика только по прошлому
    item_total = {}
    item_bots = {}

    rows = []

    # Обрабатываем одинаковый timestamp одной группой,
    # чтобы cookie из одного момента не видели target друг друга.
    for ts, batch in ordered.groupby(
        "window_start_ts",
        sort=True
    ):

        # Сначала рассчитываем признаки
        for row in batch.itertuples():
            items = item_map.get(
                row.cookie_id,
                []
            )

            totals = np.array(
                [
                    item_total.get(item, 0)
                    for item in items
                ],
                dtype=float
            )

            bots = np.array(
                [
                    item_bots.get(item, 0)
                    for item in items
                ],
                dtype=float
            )

            if len(items):
                rates = bots / np.maximum(
                    totals,
                    1
                )

                seen_mask = totals > 0

                features = {
                    "cookie_id": row.cookie_id,

                    "hist_items_seen_before":
                        int(seen_mask.sum()),

                    "hist_item_total_mean":
                        totals.mean(),

                    "hist_item_total_max":
                        totals.max(),

                    "hist_item_bot_count_mean":
                        bots.mean(),

                    "hist_item_bot_count_max":
                        bots.max(),

                    "hist_item_bot_rate_mean":
                        rates.mean(),

                    "hist_item_bot_rate_max":
                        rates.max(),

                    "hist_item_bot_rate_q75":
                        np.quantile(
                            rates,
                            .75
                        ),

                    "hist_known_item_share":
                        seen_mask.mean(),

                    "hist_bot_item_share_25":
                        (rates >= .25).mean(),

                    "hist_bot_item_share_50":
                        (rates >= .50).mean(),

                    "hist_bot_item_share_75":
                        (rates >= .75).mean()
                }

            else:
                features = {
                    "cookie_id": row.cookie_id,
                    "hist_items_seen_before": 0,
                    "hist_item_total_mean": 0,
                    "hist_item_total_max": 0,
                    "hist_item_bot_count_mean": 0,
                    "hist_item_bot_count_max": 0,
                    "hist_item_bot_rate_mean": 0,
                    "hist_item_bot_rate_max": 0,
                    "hist_item_bot_rate_q75": 0,
                    "hist_known_item_share": 0,
                    "hist_bot_item_share_25": 0,
                    "hist_bot_item_share_50": 0,
                    "hist_bot_item_share_75": 0
                }

            rows.append(features)

        # И только ПОСЛЕ расчёта обновляем историю
        for row in batch.itertuples():
            items = item_map.get(
                row.cookie_id,
                []
            )

            for item in items:
                item_total[item] = (
                    item_total.get(item, 0) + 1
                )

                item_bots[item] = (
                    item_bots.get(item, 0)
                    + int(row.target)
                )

    return pd.DataFrame(rows)

In [16]:
def build_test_item_history(
    train_meta,
    test_meta,
    events
):
    train_items = get_cookie_items(
        train_meta,
        events
    )

    test_items = get_cookie_items(
        test_meta,
        events
    )

    train_targets = train_meta[
        ["cookie_id", "target"]
    ]

    history = train_items.merge(
        train_targets,
        on="cookie_id",
        how="left"
    )

    item_stats = (
        history
        .groupby("item_id")
        .agg(
            hist_total=(
                "cookie_id",
                "nunique"
            ),
            hist_bots=(
                "target",
                "sum"
            )
        )
    )

    item_stats["hist_bot_rate"] = (
        item_stats["hist_bots"]
        /
        item_stats["hist_total"]
        .clip(lower=1)
    )

    x = test_items.merge(
        item_stats.reset_index(),
        on="item_id",
        how="left"
    )

    x["known"] = (
        x["hist_total"].notna()
    ).astype(int)

    for c in [
        "hist_total",
        "hist_bots",
        "hist_bot_rate"
    ]:
        x[c] = x[c].fillna(0)

    x["rate_ge_25"] = (
        x["hist_bot_rate"] >= .25
    ).astype(int)

    x["rate_ge_50"] = (
        x["hist_bot_rate"] >= .50
    ).astype(int)

    x["rate_ge_75"] = (
        x["hist_bot_rate"] >= .75
    ).astype(int)

    g = x.groupby("cookie_id")

    result = g.agg(
        hist_items_seen_before=("known", "sum"),

        hist_item_total_mean=("hist_total", "mean"),
        hist_item_total_max=("hist_total", "max"),

        hist_item_bot_count_mean=("hist_bots", "mean"),
        hist_item_bot_count_max=("hist_bots", "max"),

        hist_item_bot_rate_mean=("hist_bot_rate", "mean"),
        hist_item_bot_rate_max=("hist_bot_rate", "max"),
        hist_item_bot_rate_q75=(
            "hist_bot_rate",
            lambda s: s.quantile(.75)
        ),

        hist_known_item_share=("known", "mean"),

        hist_bot_item_share_25=("rate_ge_25", "mean"),
        hist_bot_item_share_50=("rate_ge_50", "mean"),
        hist_bot_item_share_75=("rate_ge_75", "mean")
    )

    result = (
        test_meta[["cookie_id"]]
        .merge(
            result.reset_index(),
            on="cookie_id",
            how="left"
        )
    )

    return result.fillna(0)

In [17]:
def build_base_features(meta, events):
    base = build_behavior_features(
        meta,
        events
    )

    timing = build_timing_features(
        meta,
        events
    )

    event_f = build_event_features(
        meta,
        events
    )

    platform_f = build_platform_features(
        meta,
        events
    )

    repetition = build_repetition_features(
        meta,
        events
    )

    sequence = build_sequence_features(
        meta,
        events
    )

    f = (
        base
        .join(timing, how="outer")
        .join(event_f, how="outer")
        .join(platform_f, how="outer")
        .join(repetition, how="outer")
        .join(sequence, how="outer")
    )

    f = add_ratio_features(f)

    out = meta[
        [
            "cookie_id",
            "cookie_created_at",
            "window_start_ts"
        ]
    ].copy()

    out["cookie_age_hours"] = (
        (
            out["window_start_ts"]
            - out["cookie_created_at"]
        )
        .dt.total_seconds()
        / 3600
    )

    out = out[[
        "cookie_id",
        "cookie_age_hours"
    ]]

    out = out.merge(
        f.reset_index(),
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )

    return out

In [18]:
print("Строим behavioral features...")

X_train = build_base_features(
    train,
    events
)

X_test = build_base_features(
    test,
    events
)

print("Строим historical item features...")

train_history = build_train_item_history(
    train,
    events
)

test_history = build_test_item_history(
    train,
    test,
    events
)

X_train = X_train.merge(
    train_history,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

X_test = X_test.merge(
    test_history,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Строим behavioral features...
Строим historical item features...
Train: (11091, 97)
Test: (4909, 97)


In [19]:
feature_cols = [
    c for c in X_train.columns
    if c != "cookie_id"
]

X_test = X_test.reindex(
    columns=[
        "cookie_id",
        *feature_cols
    ],
    fill_value=0
)

X_train[feature_cols] = (
    X_train[feature_cols]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

X_test[feature_cols] = (
    X_test[feature_cols]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

assert X_train["cookie_id"].equals(
    train["cookie_id"]
)

assert X_test["cookie_id"].equals(
    test["cookie_id"]
)

print("Количество признаков:", len(feature_cols))

Количество признаков: 96


## 7. Исторические признаки пересечения cookie и товаров

Дополнительно моделируется сходство текущего поведения с поведением ранее наблюдавшихся пользователей.

Cookie представляется множеством просмотренных `item_id`. Для текущего cookie находятся исторические cookie, имеющие с ним общие товары.

На основе этих связей рассчитываются признаки:

- число исторических соседей;
- число bot- и human-соседей;
- число соседей минимум с 2 и 3 общими товарами;
- максимальное и среднее количество общих товаров;
- доля bot-соседей;
- взвешенная доля bot-соседей с учетом числа общих товаров;
- максимальное сходство с bot и human cookie;
- Jaccard similarity между множествами просмотренных товаров.

При построении train-признаков в историю попадают только cookie с `window_end_ts <= current window_start_ts`.

Для test используется вся размеченная train-выборка, поскольку она целиком относится к более раннему временному периоду.

In [20]:
from collections import defaultdict, Counter


def build_cookie_overlap_features(meta, events):
    """
    Historical cookie-item overlap features.

    Для каждого cookie используются только cookies,
    у которых window_end_ts <= window_start_ts текущего cookie.
    """

    meta = meta.copy()

    pairs = (
        events[["cookie_id", "item_id"]]
        .dropna()
        .drop_duplicates()
        .copy()
    )

    pairs["item_id"] = (
        pairs["item_id"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    cookie_items = (
        pairs.groupby("cookie_id")["item_id"]
        .apply(set)
        .to_dict()
    )

    # item -> множество исторических cookies
    item_to_cookies = defaultdict(set)

    # target исторических cookies
    historical_target = {}

    # число items исторического cookie
    historical_n_items = {}

    rows = []

    meta_sorted = meta.sort_values(
        "window_start_ts"
    ).copy()

    # cookies, которые можно добавить в историю
    meta_by_end = meta.sort_values(
        "window_end_ts"
    ).reset_index(drop=True)

    history_pointer = 0

    for current_start, current_group in meta_sorted.groupby(
        "window_start_ts",
        sort=True
    ):

        # Добавляем только полностью завершившиеся окна
        while history_pointer < len(meta_by_end):

            hist_row = meta_by_end.iloc[history_pointer]

            if hist_row["window_end_ts"] > current_start:
                break

            hist_cookie = hist_row["cookie_id"]

            items = cookie_items.get(
                hist_cookie,
                set()
            )

            historical_target[hist_cookie] = int(
                hist_row["target"]
            )

            historical_n_items[hist_cookie] = len(items)

            for item in items:
                item_to_cookies[item].add(
                    hist_cookie
                )

            history_pointer += 1

        # Теперь считаем признаки текущих cookies
        for cookie_id in current_group["cookie_id"]:

            items = cookie_items.get(
                cookie_id,
                set()
            )

            overlap_counter = Counter()

            for item in items:
                for hist_cookie in item_to_cookies.get(
                    item,
                    ()
                ):
                    overlap_counter[hist_cookie] += 1

            if len(overlap_counter) == 0:

                rows.append({
                    "cookie_id": cookie_id,

                    "overlap_neighbors": 0,
                    "overlap_bot_neighbors": 0,
                    "overlap_human_neighbors": 0,

                    "overlap_neighbors_ge2": 0,
                    "overlap_neighbors_ge3": 0,

                    "overlap_max_shared": 0,
                    "overlap_mean_shared": 0,
                    "overlap_top3_mean": 0,

                    "overlap_bot_rate": 0,
                    "overlap_weighted_bot_rate": 0,

                    "overlap_max_bot_shared": 0,
                    "overlap_max_human_shared": 0,

                    "overlap_max_bot_jaccard": 0,
                    "overlap_max_human_jaccard": 0,

                    "overlap_bot_human_ratio": 0
                })

                continue

            overlaps = np.array(
                list(overlap_counter.values()),
                dtype=float
            )

            neighbors = list(
                overlap_counter.keys()
            )

            labels = np.array(
                [
                    historical_target[c]
                    for c in neighbors
                ],
                dtype=int
            )

            bot_mask = labels == 1
            human_mask = labels == 0

            sorted_overlaps = np.sort(
                overlaps
            )[::-1]

            weighted_bot_rate = (
                np.sum(overlaps * labels)
                / np.sum(overlaps)
            )

            bot_rate = labels.mean()

            max_bot_shared = (
                overlaps[bot_mask].max()
                if bot_mask.any()
                else 0
            )

            max_human_shared = (
                overlaps[human_mask].max()
                if human_mask.any()
                else 0
            )

            # Jaccard similarity
            bot_jaccards = []
            human_jaccards = []

            current_n = len(items)

            for hist_cookie, shared in overlap_counter.items():

                hist_n = historical_n_items[
                    hist_cookie
                ]

                union = (
                    current_n
                    + hist_n
                    - shared
                )

                jaccard = (
                    shared / union
                    if union > 0
                    else 0
                )

                if historical_target[hist_cookie] == 1:
                    bot_jaccards.append(jaccard)
                else:
                    human_jaccards.append(jaccard)

            bot_neighbors = int(
                bot_mask.sum()
            )

            human_neighbors = int(
                human_mask.sum()
            )

            rows.append({
                "cookie_id": cookie_id,

                "overlap_neighbors":
                    len(neighbors),

                "overlap_bot_neighbors":
                    bot_neighbors,

                "overlap_human_neighbors":
                    human_neighbors,

                "overlap_neighbors_ge2":
                    int((overlaps >= 2).sum()),

                "overlap_neighbors_ge3":
                    int((overlaps >= 3).sum()),

                "overlap_max_shared":
                    overlaps.max(),

                "overlap_mean_shared":
                    overlaps.mean(),

                "overlap_top3_mean":
                    sorted_overlaps[:3].mean(),

                "overlap_bot_rate":
                    bot_rate,

                "overlap_weighted_bot_rate":
                    weighted_bot_rate,

                "overlap_max_bot_shared":
                    max_bot_shared,

                "overlap_max_human_shared":
                    max_human_shared,

                "overlap_max_bot_jaccard":
                    max(bot_jaccards)
                    if bot_jaccards
                    else 0,

                "overlap_max_human_jaccard":
                    max(human_jaccards)
                    if human_jaccards
                    else 0,

                "overlap_bot_human_ratio":
                    bot_neighbors
                    / (human_neighbors + 1)
            })

    return pd.DataFrame(rows)

In [21]:
def build_test_cookie_overlap_features(
    train_meta,
    test_meta,
    events
):

    train_pairs = (
        events.loc[
            events["cookie_id"].isin(
                train_meta["cookie_id"]
            ),
            ["cookie_id", "item_id"]
        ]
        .dropna()
        .drop_duplicates()
        .copy()
    )

    test_pairs = (
        events.loc[
            events["cookie_id"].isin(
                test_meta["cookie_id"]
            ),
            ["cookie_id", "item_id"]
        ]
        .dropna()
        .drop_duplicates()
        .copy()
    )

    for df in [train_pairs, test_pairs]:
        df["item_id"] = (
            df["item_id"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    train_cookie_items = (
        train_pairs.groupby("cookie_id")["item_id"]
        .apply(set)
        .to_dict()
    )

    test_cookie_items = (
        test_pairs.groupby("cookie_id")["item_id"]
        .apply(set)
        .to_dict()
    )

    targets = dict(
        zip(
            train_meta["cookie_id"],
            train_meta["target"]
        )
    )

    item_to_cookies = defaultdict(set)

    for cookie_id, items in train_cookie_items.items():
        for item in items:
            item_to_cookies[item].add(cookie_id)

    rows = []

    for cookie_id in test_meta["cookie_id"]:

        items = test_cookie_items.get(
            cookie_id,
            set()
        )

        counter = Counter()

        for item in items:
            for hist_cookie in item_to_cookies.get(
                item,
                ()
            ):
                counter[hist_cookie] += 1

        if not counter:

            rows.append({
                "cookie_id": cookie_id,
                "overlap_neighbors": 0,
                "overlap_bot_neighbors": 0,
                "overlap_human_neighbors": 0,
                "overlap_neighbors_ge2": 0,
                "overlap_neighbors_ge3": 0,
                "overlap_max_shared": 0,
                "overlap_mean_shared": 0,
                "overlap_top3_mean": 0,
                "overlap_bot_rate": 0,
                "overlap_weighted_bot_rate": 0,
                "overlap_max_bot_shared": 0,
                "overlap_max_human_shared": 0,
                "overlap_max_bot_jaccard": 0,
                "overlap_max_human_jaccard": 0,
                "overlap_bot_human_ratio": 0
            })

            continue

        neighbors = list(counter.keys())

        overlaps = np.array(
            list(counter.values()),
            dtype=float
        )

        labels = np.array(
            [targets[c] for c in neighbors],
            dtype=int
        )

        bot_mask = labels == 1
        human_mask = labels == 0

        sorted_overlaps = np.sort(
            overlaps
        )[::-1]

        bot_jaccards = []
        human_jaccards = []

        for hist_cookie, shared in counter.items():

            hist_items = train_cookie_items[
                hist_cookie
            ]

            union = (
                len(items)
                + len(hist_items)
                - shared
            )

            jaccard = (
                shared / union
                if union > 0
                else 0
            )

            if targets[hist_cookie] == 1:
                bot_jaccards.append(jaccard)
            else:
                human_jaccards.append(jaccard)

        bot_neighbors = int(bot_mask.sum())
        human_neighbors = int(human_mask.sum())

        rows.append({
            "cookie_id": cookie_id,

            "overlap_neighbors":
                len(neighbors),

            "overlap_bot_neighbors":
                bot_neighbors,

            "overlap_human_neighbors":
                human_neighbors,

            "overlap_neighbors_ge2":
                int((overlaps >= 2).sum()),

            "overlap_neighbors_ge3":
                int((overlaps >= 3).sum()),

            "overlap_max_shared":
                overlaps.max(),

            "overlap_mean_shared":
                overlaps.mean(),

            "overlap_top3_mean":
                sorted_overlaps[:3].mean(),

            "overlap_bot_rate":
                labels.mean(),

            "overlap_weighted_bot_rate":
                np.sum(overlaps * labels)
                / np.sum(overlaps),

            "overlap_max_bot_shared":
                overlaps[bot_mask].max()
                if bot_mask.any()
                else 0,

            "overlap_max_human_shared":
                overlaps[human_mask].max()
                if human_mask.any()
                else 0,

            "overlap_max_bot_jaccard":
                max(bot_jaccards)
                if bot_jaccards
                else 0,

            "overlap_max_human_jaccard":
                max(human_jaccards)
                if human_jaccards
                else 0,

            "overlap_bot_human_ratio":
                bot_neighbors
                / (human_neighbors + 1)
        })

    return pd.DataFrame(rows)

## 8. Временная валидация

Поскольку train и test разделены по времени, случайное разбиение данных не используется.

Качество оценивается на последовательных временных интервалах. Для каждого validation-периода модель обучается только на более ранних наблюдениях.

Такой подход лучше соответствует реальному сценарию: необходимо предсказывать поведение пользователей в будущем на основе уже накопленной истории.

In [22]:
TUNING_FOLDS = [
    (
        "fold_1",
        pd.Timestamp("2026-04-09"),
        pd.Timestamp("2026-04-11")
    ),
    (
        "fold_2",
        pd.Timestamp("2026-04-11"),
        pd.Timestamp("2026-04-13")
    ),
    (
        "fold_3",
        pd.Timestamp("2026-04-13"),
        pd.Timestamp("2026-04-15")
    ),
    (
        "fold_4",
        pd.Timestamp("2026-04-15"),
        pd.Timestamp("2026-04-17")
    ),
]

HOLDOUT_START = pd.Timestamp("2026-04-17")
HOLDOUT_END = pd.Timestamp("2026-04-20")

TUNING_FOLDS

[('fold_1',
  Timestamp('2026-04-09 00:00:00'),
  Timestamp('2026-04-11 00:00:00')),
 ('fold_2',
  Timestamp('2026-04-11 00:00:00'),
  Timestamp('2026-04-13 00:00:00')),
 ('fold_3',
  Timestamp('2026-04-13 00:00:00'),
  Timestamp('2026-04-15 00:00:00')),
 ('fold_4',
  Timestamp('2026-04-15 00:00:00'),
  Timestamp('2026-04-17 00:00:00'))]

In [23]:
def prepare_fold_data(
    train_meta,
    events,
    valid_start,
    valid_end
):
    train_mask = (
        train_meta["window_start_ts"]
        < valid_start
    )

    valid_mask = (
        train_meta["window_start_ts"].ge(valid_start)
        &
        train_meta["window_start_ts"].lt(valid_end)
    )

    fold_train = (
        train_meta.loc[train_mask]
        .copy()
        .reset_index(drop=True)
    )

    fold_valid = (
        train_meta.loc[valid_mask]
        .copy()
        .reset_index(drop=True)
    )

    # 1. Behavioral features
    X_tr = build_base_features(
        fold_train,
        events
    )

    X_va = build_base_features(
        fold_valid,
        events
    )

    # 2. Historical item features
    H_tr = build_train_item_history(
        fold_train,
        events
    )

    H_va = build_test_item_history(
        fold_train,
        fold_valid,
        events
    )

    X_tr = X_tr.merge(
        H_tr,
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )

    X_va = X_va.merge(
        H_va,
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )

    # 3. NEW: historical cookie overlap features
    O_tr = build_cookie_overlap_features(
        fold_train,
        events
    )

    O_va = build_test_cookie_overlap_features(
        fold_train,
        fold_valid,
        events
    )

    X_tr = X_tr.merge(
        O_tr,
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )

    X_va = X_va.merge(
        O_va,
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )

    # 4. Список признаков
    cols = [
        c for c in X_tr.columns
        if c != "cookie_id"
    ]

    # У valid должны быть точно те же признаки
    X_va = X_va.reindex(
        columns=["cookie_id"] + cols,
        fill_value=0
    )

    # 5. Убираем inf / NaN
    X_tr[cols] = (
        X_tr[cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    X_va[cols] = (
        X_va[cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return {
        "X_train": X_tr[cols].copy(),
        "X_valid": X_va[cols].copy(),

        "y_train": fold_train["target"].to_numpy(),
        "y_valid": fold_valid["target"].to_numpy(),

        "columns": cols,

        "n_train": len(fold_train),
        "n_valid": len(fold_valid),

        "valid_start": valid_start,
        "valid_end": valid_end
    }

In [24]:
fold_data = {}

for name, start, end in TUNING_FOLDS:

    print(name, start, "→", end)

    fold_data[name] = prepare_fold_data(
        train,
        events,
        start,
        end
    )

    d = fold_data[name]

    print(
        "train:",
        d["X_train"].shape,
        "| valid:",
        d["X_valid"].shape,
        "| bots valid:",
        int(d["y_valid"].sum())
    )

print("\nВсе tuning folds готовы.")

fold_1 2026-04-09 00:00:00 → 2026-04-11 00:00:00
train: (2403, 111) | valid: (1814, 111) | bots valid: 148
fold_2 2026-04-11 00:00:00 → 2026-04-13 00:00:00
train: (4217, 111) | valid: (1713, 111) | bots valid: 146
fold_3 2026-04-13 00:00:00 → 2026-04-15 00:00:00
train: (5930, 111) | valid: (1669, 111) | bots valid: 123
fold_4 2026-04-15 00:00:00 → 2026-04-17 00:00:00
train: (7599, 111) | valid: (1541, 111) | bots valid: 125

Все tuning folds готовы.


In [25]:
def percentile_rank(values):
    return (
        pd.Series(values)
        .rank(method="average", pct=True)
        .to_numpy()
    )

## 9. Модели

В качестве базовых алгоритмов используются два градиентных бустинга:

- CatBoost;
- LightGBM.

Гиперпараметры были выбраны по результатам экспериментов на temporal cross-validation и зафиксированы перед финальной оценкой.

Итоговый прогноз представляет собой rank-based ensemble:

`0.60 × CatBoost + 0.40 × LightGBM`

Перед объединением предсказания каждой модели преобразуются в percentile ranks. Это уменьшает влияние различий в калибровке вероятностей двух моделей.

In [26]:
CAT_FINALISTS = {
    "cat_H": {
        "iterations": 1100,
        "depth": 6,
        "learning_rate": 0.025,
        "l2_leaf_reg": 10
    }
}

LGB_FINALISTS = {
    "lgb_E": {
        "n_estimators": 700,
        "learning_rate": 0.025,
        "num_leaves": 63,
        "max_depth": 7,
        "min_child_samples": 40,
        "colsample_bytree": 0.65,
        "subsample": 0.85,
        "reg_alpha": 0.15,
        "reg_lambda": 0.80
    }
}

FINAL_CAT_PARAMS = CAT_FINALISTS["cat_H"]
FINAL_LGB_PARAMS = LGB_FINALISTS["lgb_E"]

FINAL_CAT_WEIGHT = 0.60
FINAL_LGB_WEIGHT = 0.40

print("Final CatBoost:", FINAL_CAT_PARAMS)
print("Final LightGBM:", FINAL_LGB_PARAMS)
print("Weights:", FINAL_CAT_WEIGHT, FINAL_LGB_WEIGHT)

Final CatBoost: {'iterations': 1100, 'depth': 6, 'learning_rate': 0.025, 'l2_leaf_reg': 10}
Final LightGBM: {'n_estimators': 700, 'learning_rate': 0.025, 'num_leaves': 63, 'max_depth': 7, 'min_child_samples': 40, 'colsample_bytree': 0.65, 'subsample': 0.85, 'reg_alpha': 0.15, 'reg_lambda': 0.8}
Weights: 0.6 0.4


In [27]:
experiment_rows = []

for fold_name, d in fold_data.items():

    cat_model = CatBoostClassifier(
        **CAT_FINALISTS["cat_H"],
        loss_function="Logloss",
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=4
    )

    cat_model.fit(
        d["X_train"],
        d["y_train"]
    )

    cat_pred = cat_model.predict_proba(
        d["X_valid"]
    )[:, 1]


    lgb_model = lgb.LGBMClassifier(
        **LGB_FINALISTS["lgb_E"],
        objective="binary",
        random_state=SEED,
        n_jobs=4,
        verbosity=-1,
        subsample_freq=1
    )

    lgb_model.fit(
        d["X_train"],
        d["y_train"]
    )

    lgb_pred = lgb_model.predict_proba(
        d["X_valid"]
    )[:, 1]


    blend_pred = (
        0.60 * percentile_rank(cat_pred)
        +
        0.40 * percentile_rank(lgb_pred)
    )

    experiment_rows.append({
        "fold": fold_name,

        "CatBoost":
            precision_at_recall(
                d["y_valid"],
                cat_pred
            ),

        "LightGBM":
            precision_at_recall(
                d["y_valid"],
                lgb_pred
            ),

        "Blend":
            precision_at_recall(
                d["y_valid"],
                blend_pred
            )
    })


overlap_results = pd.DataFrame(
    experiment_rows
)

display(overlap_results)

print(
    "\nMean Blend:",
    overlap_results["Blend"].mean()
)

print(
    "Min Blend:",
    overlap_results["Blend"].min()
)

print(
    "Features:",
    fold_data["fold_1"]["X_train"].shape[1]
)

,fold,CatBoost,LightGBM,Blend
0,fold_1,0.835938,0.832000,0.859504
1,fold_2,0.896552,0.903509,0.895652
2,fold_3,0.956989,0.936170,0.957447
3,fold_4,0.918367,0.928571,0.936842



Mean Blend: 0.9123613049795611
Min Blend: 0.859504132231405
Features: 111


### Результат temporal validation

Добавление historical cookie-item overlap признаков улучшило качество модели на всех временных разбиениях.

Средний `Precision@Recall>=0.70` для финального ансамбля составляет около **0.912**.

Это указывает на то, что информация о пересечениях просмотренных товаров между текущими и историческими cookie содержит полезный сигнал для детекции автоматизированного поведения.

## 10. Holdout validation

После выбора признаков, моделей и весов ансамбля выполняется дополнительная проверка на более позднем временном периоде — 17–19 апреля.

Этот период не используется для подбора гиперпараметров или весов ансамбля и служит финальной проверкой способности решения переноситься на будущие данные.

In [28]:
holdout_data = prepare_fold_data(
    train,
    events,
    pd.Timestamp("2026-04-17"),
    pd.Timestamp("2026-04-20")
)

print(
    "Train:",
    holdout_data["X_train"].shape
)

print(
    "Holdout:",
    holdout_data["X_valid"].shape
)

print(
    "Bots in holdout:",
    int(holdout_data["y_valid"].sum())
)

Train: (9140, 111)
Holdout: (1951, 111)
Bots in holdout: 160


In [29]:
holdout_cat = CatBoostClassifier(
    **FINAL_CAT_PARAMS,
    loss_function="Logloss",
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=4
)

holdout_cat.fit(
    holdout_data["X_train"],
    holdout_data["y_train"]
)

holdout_cat_pred = holdout_cat.predict_proba(
    holdout_data["X_valid"]
)[:, 1]


holdout_lgb = lgb.LGBMClassifier(
    **FINAL_LGB_PARAMS,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1,
    subsample_freq=1
)

holdout_lgb.fit(
    holdout_data["X_train"],
    holdout_data["y_train"]
)

holdout_lgb_pred = holdout_lgb.predict_proba(
    holdout_data["X_valid"]
)[:, 1]

In [30]:
holdout_blend = (
    FINAL_CAT_WEIGHT
    * percentile_rank(holdout_cat_pred)
    +
    FINAL_LGB_WEIGHT
    * percentile_rank(holdout_lgb_pred)
)

holdout_scores = pd.DataFrame({
    "model": [
        "CatBoost",
        "LightGBM",
        "Selected blend"
    ],

    "P@R>=0.70": [
        precision_at_recall(
            holdout_data["y_valid"],
            holdout_cat_pred
        ),

        precision_at_recall(
            holdout_data["y_valid"],
            holdout_lgb_pred
        ),

        precision_at_recall(
            holdout_data["y_valid"],
            holdout_blend
        )
    ]
})

display(holdout_scores)

,model,P@R>=0.70
0,CatBoost,0.881890
1,LightGBM,0.877863
2,Selected blend,0.881890


Финальный ансамбль показывает `Precision@Recall>=0.70` около **0.882** на holdout-периоде, что подтверждает переносимость выбранного подхода на более поздние данные.

## 11. Обучение финальной модели

После завершения валидации historical overlap признаки строятся для всей обучающей и тестовой выборок.

Итоговый набор содержит **111 признаков**.

CatBoost и LightGBM переобучаются на всей доступной размеченной выборке, после чего их test-предсказания объединяются с зафиксированными весами 0.60 / 0.40.

In [31]:
print("Строим overlap features для полного train...")

O_train_full = build_cookie_overlap_features(
    train,
    events
)

print("Строим overlap features для test...")

O_test_full = build_test_cookie_overlap_features(
    train,
    test,
    events
)

print(
    "Overlap train:",
    O_train_full.shape
)

print(
    "Overlap test:",
    O_test_full.shape
)

Строим overlap features для полного train...
Строим overlap features для test...
Overlap train: (11091, 16)
Overlap test: (4909, 16)


In [32]:
X_train_111 = X_train.merge(
    O_train_full,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

X_test_111 = X_test.merge(
    O_test_full,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

feature_cols_111 = [
    c for c in X_train_111.columns
    if c != "cookie_id"
]

X_test_111 = X_test_111.reindex(
    columns=["cookie_id"] + feature_cols_111,
    fill_value=0
)

X_train_111[feature_cols_111] = (
    X_train_111[feature_cols_111]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

X_test_111[feature_cols_111] = (
    X_test_111[feature_cols_111]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    "Train:",
    X_train_111.shape
)

print(
    "Test:",
    X_test_111.shape
)

print(
    "Features:",
    len(feature_cols_111)
)

Train: (11091, 112)
Test: (4909, 112)
Features: 111


In [33]:
X_full = X_train_111[
    feature_cols_111
].copy()

X_final_test = X_test_111[
    feature_cols_111
].copy()

y_full = train["target"].to_numpy()

print("Train:", X_full.shape)
print("Test:", X_final_test.shape)
print("Bots:", y_full.sum())

Train: (11091, 111)
Test: (4909, 111)
Bots: 899


In [34]:
final_cat = CatBoostClassifier(
    **FINAL_CAT_PARAMS,
    loss_function="Logloss",
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=4
)

final_cat.fit(
    X_full,
    y_full
)

test_cat_pred = final_cat.predict_proba(
    X_final_test
)[:, 1]


final_lgb = lgb.LGBMClassifier(
    **FINAL_LGB_PARAMS,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1,
    subsample_freq=1
)

final_lgb.fit(
    X_full,
    y_full
)

test_lgb_pred = final_lgb.predict_proba(
    X_final_test
)[:, 1]

print("Models trained on:", X_full.shape[1], "features")
print("Predictions:", len(test_cat_pred), len(test_lgb_pred))

Models trained on: 111 features
Predictions: 4909 4909


In [35]:
final_score = (
    FINAL_CAT_WEIGHT
    * percentile_rank(test_cat_pred)
    +
    FINAL_LGB_WEIGHT
    * percentile_rank(test_lgb_pred)
)

## 12. Формирование submission

Финальные значения `score` сопоставляются с `cookie_id` из `sample_submission.csv`.

Перед сохранением проверяются:

- количество строк;
- уникальность `cookie_id`;
- отсутствие пропущенных значений;
- диапазон `score` от 0 до 1;
- соответствие множества cookie тестовой выборке.

In [36]:
score_by_cookie = pd.Series(
    final_score,
    index=test["cookie_id"]
)

submission = sample_submission[
    ["cookie_id"]
].copy()

submission["score"] = (
    submission["cookie_id"]
    .map(score_by_cookie)
)

assert len(submission) == 4909
assert submission["cookie_id"].is_unique
assert submission["score"].notna().all()
assert submission["score"].between(0, 1).all()

print("Submission validation: OK")
display(submission.head())

Submission validation: OK


,cookie_id,score
0,ck_99a4e5ef02d89493,0.761336
1,ck_54d463f7fd89ec3d,0.545895
2,ck_0fbbc7a6af300368,0.577429
3,ck_8fd4937eadd64fb6,0.228682
4,ck_dbb4bd4eda97255d,0.539214


In [37]:
SUBMISSION_PATH = ROOT / "submission.csv"

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

print("Saved:", SUBMISSION_PATH)
print("Rows:", len(submission))

Saved: c:\test_task_avito\bot_detection\submission.csv
Rows: 4909


## 13. Вывод

В работе было построено решение для детекции автоматизированного поведения пользователей на основе событий внутри временного окна.

Основу решения составили поведенческие и временные характеристики активности, статистики взаимодействия с товарами, а также исторические признаки. Дополнительно были построены признаки сходства текущих cookie с ранее наблюдавшимися пользователями через пересечения по `item_id`.

Для корректной оценки качества использовалось временное разбиение данных, при котором при построении исторических признаков учитывалась только информация из прошлого. Это позволило избежать утечки целевой переменной из будущих периодов.

Финальное представление содержит **111 признаков**. В качестве моделей использовались CatBoost и LightGBM, а итоговый прогноз формировался как rank-based ансамбль с весами **0.60 и 0.40** соответственно.

Среднее значение `Precision@Recall≥0.70` на temporal validation составило около **0.912**, а на отдельном holdout-периоде — около **0.882**.

В результате после обучения моделей на всей доступной обучающей выборке был сформирован файл `submission.csv` с предсказаниями для 4 909 объектов тестовой выборки.